# Guide to using Family-level coxpression for Stress Prediction
## Step 1: Generate Gene Level differential expression data for both species to compare our method to

In [ ]:
import pandas as pd
from pydeseq2.dds import DeseqDataSet
from pydeseq2.default_inference import DefaultInference
from pydeseq2.ds import DeseqStats
from pydeseq2.utils import load_example_data
import matplotlib.pyplot as plt
import tqdm as tq
import numpy as np 
import seaborn as sns

In [ ]:
your_data_directory = "C:/Users/mikep/git/"
arabi_csv_uncleaned = pd.read_csv(
    f"{your_data_directory}/Family_level_coexpression/Data_for_tutorial/Arabi_tomato_comparison/GSE94015_gene_readCount.txt",
    sep="\t",
)
arabi_csv = arabi_csv_uncleaned[
    [
        "Locus",
        "WtRL0h_rep1",
        "WtRL0h_rep2",
        "WtRL0h_rep3",
        "WtRL3h_rep1",
        "WtRL3h_rep2",
        "WtRL3h_rep3",
    ]
]

arabi_csv = arabi_csv.set_index('Locus')
arabi_csv


In [ ]:
arabi_metadata = pd.DataFrame(
    columns=["Temp"],
    data=["Cold", "Cold", "Cold", "Hot", "Hot", "Hot"],
    index=[
        "WtRL0h_rep1",
        "WtRL0h_rep2",
        "WtRL0h_rep3",
        "WtRL3h_rep1",
        "WtRL3h_rep2",
        "WtRL3h_rep3",
    ],
)

In [ ]:
low_heat_tomato_1 = pd.read_csv(
    f"{your_data_directory}/Family_level_coexpression/Data_for_tutorial/Arabi_tomato_comparison/low_heat_tomato/tomato_no_heat_stress_1ReadsPerGene.out.tab",
    sep="\t",
    index_col=0,
    skiprows=4,
    header=None,
    names=["Reads", "Left Reads", "Right reads"],
)
low_heat_tomato_2 = pd.read_csv(
    f"{your_data_directory}/Family_level_coexpression/Data_for_tutorial/Arabi_tomato_comparison/low_heat_tomato/tomato_no_heat_stress_2ReadsPerGene.out.tab",
    sep="\t",
    index_col=0,
    skiprows=4,
    header=None,
    names=["Reads", "Left Reads", "Right reads"],
)
low_heat_tomato_3 = pd.read_csv(
    f"{your_data_directory}/Family_level_coexpression/Data_for_tutorial/Arabi_tomato_comparison/low_heat_tomato/tomato_no_heat_stress_3ReadsPerGene.out.tab",
    sep="\t",
    index_col=0,
    skiprows=4,
    header=None,
    names=["Reads", "Left Reads", "Right reads"],
)

low_heat_tomato = pd.DataFrame(
    data=zip(
        low_heat_tomato_1["Reads"],
        low_heat_tomato_2["Reads"],
        low_heat_tomato_3["Reads"],
    ),
    columns=["Tom Cold Rep 1", "Tom Cold Rep 2", "Tom Cold Rep 3"],
    index=low_heat_tomato_1.index,
)

In [ ]:
high_heat_tomato_rep_1 = pd.read_csv(
    f"{your_data_directory}/Family_level_coexpression/Data_for_tutorial/Arabi_tomato_comparison/high_heat_tomato/tomato_heat_stress_1ReadsPerGene.out.tab",
    sep="\t",
    index_col=0,
    skiprows=4,
    header=None,
    names=["Reads", "Left Reads", "Right reads"],
)
high_heat_tomato_rep_2 = pd.read_csv(
   f"{your_data_directory}/Family_level_coexpression/Data_for_tutorial/Arabi_tomato_comparison/high_heat_tomato/tomato_heat_stress_2ReadsPerGene.out.tab",
    sep="\t",
    index_col=0,
    skiprows=4,
    header=None,
    names=["Reads", "Left Reads", "Right reads"],
)
high_heat_tomato_rep_3 = pd.read_csv(
    f"{your_data_directory}/Family_level_coexpression/Data_for_tutorial/Arabi_tomato_comparison/high_heat_tomato/tomato_heat_stress_3ReadsPerGene.out.tab",
    sep="\t",
    index_col=0,
    skiprows=4,
    header=None,
    names=["Reads", "Left Reads", "Right reads"],
)
high_heat_tomato = pd.DataFrame(
    data=zip(
        high_heat_tomato_rep_1["Reads"],
        high_heat_tomato_rep_2["Reads"],
        high_heat_tomato_rep_3["Reads"],
    ),
    columns=["Tom Hot Rep 1", "Tom Hot Rep 2", "Tom Hot Rep 3"],
    index=low_heat_tomato_1.index,
)

In [ ]:
combined_tomato = pd.concat([high_heat_tomato, low_heat_tomato], axis=1)

In [ ]:
combined_tomato = combined_tomato.T
arabi_csv = arabi_csv.T


In [ ]:
tomato_metadata = pd.DataFrame(
    columns=["Temp"],
    data=["Hot", "Hot", "Hot", "Cold", "Cold", "Cold"],
    index=[
        "Tom Hot Rep 1",
        "Tom Hot Rep 2",
        "Tom Hot Rep 3",
        "Tom Cold Rep 1",
        "Tom Cold Rep 2",
        "Tom Cold Rep 3",
    ],
)

In [ ]:
genes_to_keep_arabi = arabi_csv.columns[arabi_csv.sum(axis=0) >= 100]
arabi_csv = arabi_csv[genes_to_keep_arabi]
inference_arabi = DefaultInference(n_cpus=30)
dds_arabi = DeseqDataSet(
    counts=arabi_csv,
    metadata=arabi_metadata,
    design_factors="Temp",
    refit_cooks=True,
    inference=inference_arabi,
)
dds_arabi.deseq2()
stat_res_arabi = DeseqStats(dds_arabi, inference=inference_arabi)
stat_res_arabi.summary()

In [ ]:
genes_to_keep = combined_tomato.columns[combined_tomato.sum(axis=0) >= 100]
combined_tomato = combined_tomato[genes_to_keep]
inference = DefaultInference(n_cpus=30)
dds = DeseqDataSet(
    counts=combined_tomato,
    metadata=tomato_metadata,
    design_factors="Temp",
    refit_cooks=True,
    inference=inference,
)
dds.deseq2()
stat_res = DeseqStats(dds, inference=inference)
stat_res.summary()

In [ ]:
full_results = stat_res.results_df
full_results
full_arabi_results = stat_res_arabi.results_df

## Step 2: Do the family-level differential expression 

In [ ]:
def species_name_resolver(species_1,desired_type = 'common'):
    """[Takes ambiguous form of species name and returns desired type]

    Args:
        species_1 ([str]): [Ambigious Species Name]
        desired_type (str, optional): [One of common, scientific, or taxa_id]. Defaults to 'common'.

    Returns:
        [str]: [Specified Species ID]
    """

    import pandas as pd
    
    # Assert 
    assert (desired_type in ['common','scientific','taxa_id']), 'Desired type should be common, scientific, or taxa_id'
   
    #Set up variable 
    fc_mapper = pd.read_csv(f'{your_data_directory}/Family_level_coexpression/Data_for_tutorial/Species_name_resolver.csv')

    #Convert Taxa to common names if NCBI taxa ID
    if type(species_1) == int:
        species_1 = fc_mapper['Common Name'].loc[fc_mapper['Taxa ID'] == species_1].item()

    #Convert scientific name to common names if given scientific
    if ' ' in species_1:
        species_1 = fc_mapper['Common Name'].loc[fc_mapper['Species'] == species_1].item()

    #Get Scientific Name
    scientific_1 = fc_mapper['Species'].loc[fc_mapper['Common Name'] == species_1].item()
    taxa_id_1 = fc_mapper['Taxa ID'].loc[fc_mapper['Common Name'] == species_1].item()

    if desired_type == 'common':
        return species_1
    elif desired_type =='scientific':
        return scientific_1
    elif desired_type == 'Taxa ID':
        return taxa_id_1

In [ ]:
def get_ncbi_clean_og2gene_for_species(
    species_1, og2genes_only_cococonet, ncbi_mapping
):

    species_1_name = species_name_resolver(
        species_1, desired_type="common"
    )

    first_species_ortho_groups = og2genes_only_cococonet.loc[
        og2genes_only_cococonet["Species"] == species_1
    ]
    shared_orthogroups = first_species_ortho_groups["Orthogroup"].unique()

    list_of_orthogene_pds = []
    for orthogroup in tq.tqdm(
        shared_orthogroups, desc="inner_loop", position=0, leave=False
    ):
        species_1_genes = (
            first_species_ortho_groups["Gene"]
            .loc[first_species_ortho_groups["Orthogroup"] == orthogroup]
            .to_list()
        )
        all_gene_combos = species_1_genes
        current_orthogroup_pd = pd.DataFrame(
            columns=[f"{species_1_name} OrthoGene"], data=all_gene_combos
        )
        current_orthogroup_pd["Orthogroup"] = orthogroup
        list_of_orthogene_pds.append(current_orthogroup_pd)

    final_species_lineup = pd.concat(list_of_orthogene_pds)
    ncbi_added_once = final_species_lineup.merge(
        right=ncbi_mapping[["Orthodb Gene", "Symbol"]],
        right_on="Orthodb Gene",
        left_on=f"{species_1_name} OrthoGene",
    )
    ncbi_added_once_clean = ncbi_added_once.drop(columns="Orthodb Gene")
    return ncbi_added_once_clean

In [ ]:
og_groups = pd.read_csv(
    f"{your_data_directory}/Family_level_coexpression/Data_for_tutorial/og_2_Genes_with_ncbi_symbol.csv"
)
og_groups

In [ ]:
ncbi_mapping = pd.read_csv(
    f"{your_data_directory}/Family_level_coexpression/Data_for_tutorial/merged_ncbi_to_orthodb_fixed_non_genesymbol.csv"
)

In [ ]:
tomato_og = get_ncbi_clean_og2gene_for_species(
    4081, og_groups, ncbi_mapping=ncbi_mapping
)

In [ ]:
arabi_og = get_ncbi_clean_og2gene_for_species(
    3702, og_groups, ncbi_mapping=ncbi_mapping
)

In [ ]:
tomato_og

In [ ]:
list_of_non_single_gene_groups_tomato = pd.Series(tomato_og['Orthogroup'].value_counts().loc[tomato_og['Orthogroup'].value_counts() >2])

In [ ]:
list_of_non_single_gene_groups_arabi = pd.Series(arabi_og['Orthogroup'].value_counts().loc[arabi_og['Orthogroup'].value_counts() >2])

In [ ]:
list_of_non_single_gene_groups_arabi

In [ ]:
list_of_non_single_gene_groups_tomato

In [ ]:
tomato_og = tomato_og.loc[tomato_og['Orthogroup'].isin(list_of_non_single_gene_groups_tomato.index)]

In [ ]:
arabi_og = arabi_og.loc[arabi_og['Orthogroup'].isin(list_of_non_single_gene_groups_arabi.index)]

In [ ]:
combined_arabi_divert = arabi_csv.copy()

In [ ]:
dict_version = arabi_og[['Orthogroup','Symbol']].set_index('Symbol').to_dict()
true_dict = dict_version['Orthogroup']
arabi_csv = arabi_csv.T.groupby(by = true_dict).mean()

In [ ]:
arabi_csv

In [ ]:
combined_tomato_divert = combined_tomato.copy()

In [ ]:
combined_tomato

In [ ]:
dict_version = tomato_og[['Orthogroup','Symbol']].set_index('Symbol').to_dict()
true_dict = dict_version['Orthogroup']
combined_tomato = combined_tomato.T.groupby(by = true_dict).mean()

In [ ]:
combined_tomato

In [ ]:
combined_tomato = combined_tomato.T
arabi_csv = arabi_csv.T

In [ ]:
combined_tomato = combined_tomato.round(0)
arabi_csv = arabi_csv.round(0)

In [ ]:
genes_to_keep_arabi_ortho = arabi_csv.columns[arabi_csv.sum(axis=0) >= 100]
arabi_csv_ortho = arabi_csv[genes_to_keep_arabi_ortho]
inference_arabi_ortho = DefaultInference(n_cpus=30)
dds_arabi_ortho = DeseqDataSet(
    counts=arabi_csv_ortho,
    metadata=arabi_metadata,
    design_factors="Temp",
    refit_cooks=True,
    inference=inference_arabi_ortho,
)
dds_arabi_ortho.deseq2()
stat_res_arabi_ortho = DeseqStats(dds_arabi_ortho, inference=inference_arabi_ortho)
stat_res_arabi_ortho.summary()

In [ ]:
genes_to_keep_tom_ortho = combined_tomato.columns[combined_tomato.sum(axis=0) >= 100]
combined_tomato_ortho = combined_tomato[genes_to_keep_tom_ortho]
inference_tom_ortho = DefaultInference(n_cpus=30)
dds_tom_ortho = DeseqDataSet(
    counts=combined_tomato_ortho,
    metadata=tomato_metadata,
    design_factors="Temp",
    refit_cooks=True,
    inference=inference_tom_ortho,
)
dds_tom_ortho.deseq2()
stat_res_tom_ortho = DeseqStats(dds_tom_ortho, inference=inference_tom_ortho)
stat_res_tom_ortho.summary()

##Step 3: Compare results

In [ ]:
arabi_genes = stat_res_arabi.results_df
tomato_genes = stat_res.results_df
arabi_ortho = stat_res_arabi_ortho.results_df
tomato_ortho = stat_res_tom_ortho.results_df

In [ ]:
log_fc_change_evaluation_value = 1.5
log_fc_change_evaluation_value_negative = -1*log_fc_change_evaluation_value

In [ ]:
trimmed_tomato_ortho = tomato_ortho.loc[tomato_ortho.index.isin(arabi_ortho.index)]
trimmed_arabi_ortho = arabi_ortho.loc[arabi_ortho.index.isin(trimmed_tomato_ortho.index)]

In [ ]:
trimmed_arabi_ortho_named = trimmed_arabi_ortho.rename(columns = {'log2FoldChange':'Arabidopsis Log2FC'})
trimmed_tomato_ortho_named = trimmed_tomato_ortho.rename(columns = {'log2FoldChange':'Tomato Log2FC'})


In [ ]:
merged_ortho_df = trimmed_arabi_ortho_named[['Arabidopsis Log2FC']].merge(right = trimmed_tomato_ortho_named, left_index= True, right_index = True)

In [ ]:
merged_ortho_df

In [ ]:
merged_result = pd.DataFrame(data = zip(trimmed_tomato_ortho['log2FoldChange'], trimmed_arabi_ortho['log2FoldChange']), columns = ['Tomato','Arabi'])

all_over_1 = merged_result.loc[(merged_result['Tomato']>log_fc_change_evaluation_value) & (merged_result['Arabi']>log_fc_change_evaluation_value)]
all_under_1 = merged_result.loc[(merged_result['Tomato']<log_fc_change_evaluation_value_negative) & (merged_result['Arabi']<log_fc_change_evaluation_value_negative)]
discord_bottom = merged_result.loc[(merged_result['Tomato']>log_fc_change_evaluation_value) & (merged_result['Arabi']<log_fc_change_evaluation_value_negative)]
discord_top = merged_result.loc[(merged_result['Tomato']<log_fc_change_evaluation_value_negative) & (merged_result['Arabi']>log_fc_change_evaluation_value)]

In [ ]:
merged_result['Classification'] = 'Non-substantial'
merged_result['Classification'].loc[(merged_result['Tomato']>log_fc_change_evaluation_value) & (merged_result['Arabi']>log_fc_change_evaluation_value)] = 'Substantial'
merged_result['Classification'].loc[(merged_result['Tomato']<log_fc_change_evaluation_value_negative) & (merged_result['Arabi']<log_fc_change_evaluation_value_negative)]= 'Substantial'
merged_result['Classification'].loc[(merged_result['Tomato']>log_fc_change_evaluation_value) & (merged_result['Arabi']<log_fc_change_evaluation_value_negative)]= 'Substantial'
merged_result['Classification'].loc[(merged_result['Tomato']<log_fc_change_evaluation_value_negative) & (merged_result['Arabi']>log_fc_change_evaluation_value)]= 'Substantial'

In [ ]:
discord_ratio = (len(all_over_1)+len(all_under_1))/(len(discord_bottom)+len(discord_top))
discord_ratio

In [ ]:
sns.scatterplot(data = merged_result.loc[merged_result['Classification'] == "Substantial"] , x ='Tomato', y = 'Arabi', s = 10)
plt.axvline(0, color="k", linestyle="--")
plt.axhline(0, color="k", linestyle="--")
plt.axvline(1.5, color="grey", linestyle="--")
plt.axhline(1.5, color="grey", linestyle="--")
plt.axvline(-1.5, color="grey", linestyle="--")
plt.axhline(-1.5, color="grey", linestyle="--")
plt.title('Orthogroupwise Tomato vs Arabi \n 4.42 Concordant Groups per Discordant Group')
plt.xlabel("Orthogroup-wise log2-fold change in Tomato", fontsize = 12)
plt.ylabel("Orthogroup-wise log2-fold change in Arabidopsis", fontsize = 12)

In [ ]:
arabidopsis_tomato_nm = pd.read_csv(f'{your_data_directory}Family_level_coexpression/Data_for_tutorial/tomato_to_arabidopsis_ortholog_NM.csv')

In [ ]:
arabidopsis_tomato_nm = arabidopsis_tomato_nm.drop_duplicates(subset = 'arabidopsis Symbol',)
arabidopsis_tomato_nm = arabidopsis_tomato_nm.drop_duplicates(subset = 'tomato Symbol',)
arabidopsis_tomato_nm

In [ ]:
arabi_genes_trimmed = arabi_genes.merge(how = 'inner', left_index= True, right = arabidopsis_tomato_nm[['tomato Symbol','arabidopsis Symbol']], right_on= 'arabidopsis Symbol')
arabi_genes_trimmed = arabi_genes_trimmed.drop_duplicates(subset = 'tomato Symbol', keep = False)
arabi_genes_trimmed

In [ ]:
tomato_genes_trimmed = tomato_genes.merge(how = 'inner', left_index= True, right = arabidopsis_tomato_nm[['tomato Symbol','arabidopsis Symbol']], right_on= 'tomato Symbol')
tomato_genes_trimmed = tomato_genes_trimmed.drop_duplicates(subset = 'arabidopsis Symbol', keep = False)
tomato_genes_trimmed

In [ ]:
arabi_genes_trimmed = arabi_genes_trimmed.loc[arabi_genes_trimmed['arabidopsis Symbol'].isin(tomato_genes_trimmed['arabidopsis Symbol'])]
tomato_genes_trimmed = tomato_genes_trimmed.loc[tomato_genes_trimmed['tomato Symbol'].isin(arabi_genes_trimmed['tomato Symbol'])]

In [ ]:
tomato_genes_trimmed = (tomato_genes_trimmed.set_index('arabidopsis Symbol')
          .reindex(arabi_genes_trimmed.set_index('arabidopsis Symbol').index)
          .reset_index()
       )

In [ ]:
arabi_genes_trimmed = arabi_genes_trimmed.reset_index()

In [ ]:
new_merged_result = pd.DataFrame(data = zip(tomato_genes_trimmed['log2FoldChange'], arabi_genes_trimmed['log2FoldChange']), columns = ['Tomato','Arabi'])

all_over_1 = new_merged_result.loc[(new_merged_result['Tomato']>log_fc_change_evaluation_value) & (new_merged_result['Arabi']>log_fc_change_evaluation_value)]
all_under_1 = new_merged_result.loc[(new_merged_result['Tomato']<log_fc_change_evaluation_value_negative) & (new_merged_result['Arabi']<log_fc_change_evaluation_value_negative)]
discord_bottom = new_merged_result.loc[(new_merged_result['Tomato']>log_fc_change_evaluation_value) & (new_merged_result['Arabi']<log_fc_change_evaluation_value_negative)]
discord_top = new_merged_result.loc[(new_merged_result['Tomato']<log_fc_change_evaluation_value_negative) & (new_merged_result['Arabi']>log_fc_change_evaluation_value)]

In [ ]:
new_merged_result['Classification'] = 'Non-substantial'
new_merged_result['Classification'].loc[(new_merged_result['Tomato']>log_fc_change_evaluation_value) & (new_merged_result['Arabi']>log_fc_change_evaluation_value)] = 'Substantial'
new_merged_result['Classification'].loc[(new_merged_result['Tomato']<log_fc_change_evaluation_value_negative) & (new_merged_result['Arabi']<log_fc_change_evaluation_value_negative)]= 'Substantial'
new_merged_result['Classification'].loc[(new_merged_result['Tomato']>log_fc_change_evaluation_value) & (new_merged_result['Arabi']<log_fc_change_evaluation_value_negative)]= 'Substantial'
new_merged_result['Classification'].loc[(new_merged_result['Tomato']<log_fc_change_evaluation_value_negative) & (new_merged_result['Arabi']>log_fc_change_evaluation_value)]= 'Substantial'

In [ ]:
sns.scatterplot(data = new_merged_result.loc[new_merged_result['Classification'] == "Substantial"] , x ='Tomato', y = 'Arabi', s = 10 )
plt.axvline(0, color="k", linestyle="--")
plt.axhline(0, color="k", linestyle="--")
plt.axvline(1.5, color="grey", linestyle="--")
plt.axhline(1.5, color="grey", linestyle="--")
plt.axvline(-1.5, color="grey", linestyle="--")
plt.axhline(-1.5, color="grey", linestyle="--")
plt.title('Genewise Tomato vs Arabidopsis \n2.98 Concordant Genes per Discordant Gene')
plt.ylabel("Genewise log2-fold change in Arabidopsis", fontsize = 12)
plt.xlabel("Genewise log2-fold change in Tomato", fontsize = 12)

In [ ]:
discord_ratio = (len(all_over_1)+len(all_under_1))/(len(discord_bottom)+len(discord_top))
discord_ratio

## Step 4: Compare ortho and genewise, showing improvement from 2.98 to 4.42 concordant genes per discordant gene

In [ ]:
sns.scatterplot(data = merged_result.loc[merged_result['Classification'] == "Substantial"] , x ='Tomato', y = 'Arabi', s = 10)
plt.axvline(0, color="k", linestyle="--")
plt.axhline(0, color="k", linestyle="--")
plt.axvline(1.5, color="grey", linestyle="--")
plt.axhline(1.5, color="grey", linestyle="--")
plt.axvline(-1.5, color="grey", linestyle="--")
plt.axhline(-1.5, color="grey", linestyle="--")
plt.title('Orthogroupwise Tomato vs Arabi \n 4.42 Concordant Groups per Discordant Group')
plt.xlabel("Orthogroup-wise log2-fold change in Tomato", fontsize = 12)
plt.ylabel("Orthogroup-wise log2-fold change in Arabidopsis", fontsize = 12)

In [ ]:
sns.scatterplot(data = new_merged_result.loc[new_merged_result['Classification'] == "Substantial"] , x ='Tomato', y = 'Arabi', s = 10 )
plt.axvline(0, color="k", linestyle="--")
plt.axhline(0, color="k", linestyle="--")
plt.axvline(1.5, color="grey", linestyle="--")
plt.axhline(1.5, color="grey", linestyle="--")
plt.axvline(-1.5, color="grey", linestyle="--")
plt.axhline(-1.5, color="grey", linestyle="--")
plt.title('Genewise Tomato vs Arabidopsis \n2.98 Concordant Genes per Discordant Gene')
plt.ylabel("Genewise log2-fold change in Arabidopsis", fontsize = 12)
plt.xlabel("Genewise log2-fold change in Tomato", fontsize = 12)

Some notes on version issues: You may want to use numpy 1.24.3, pandas 1.5.3, anndata 0.9.1, pydeseq2 0.4.4